# ML-06 — Signal Audit: Do the Flags Hold?

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.


## 1. Distributions

I first inspect the distributions of the main signals that may be useful for the content decision rule. I focus on search volume, content staleness, CTR, and average position.

The distributions help identify whether the signals have useful variation or whether they are heavily concentrated in a small number of values. I will use these observations before deciding whether a signal is reliable enough for the later rule.

In [1]:
# 1. Distributions

import pandas as pd
import numpy as np
from IPython.display import display

# Load the repository CSV
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Clean column names
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
print("\nKey columns available:")
print([
    "search_volume",
    "days_since_last_update",
    "freshness_tier",
    "ctr",
    "avg_position",
    "impressions_90d",
    "clicks_90d"
])

# Select the main numeric signals
distribution_cols = [
    "search_volume",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

display(df[distribution_cols].describe().T)

# Show a few percentile values to see the spread
percentiles = df[distribution_cols].quantile(
    [0.25, 0.50, 0.75, 0.90]
).T

print("\nSelected percentiles:")
display(percentiles)

Dataset shape: (30000, 44)

Key columns available:
['search_volume', 'days_since_last_update', 'freshness_tier', 'ctr', 'avg_position', 'impressions_90d', 'clicks_90d']


,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0



Selected percentiles:


,0.25,0.50,0.75,0.90
search_volume,0.0,10.00,20.00,110.00
days_since_last_update,20.0,20.00,104.00,104.00
ctr,0.0,0.07,0.29,0.65
avg_position,6.2,10.80,22.30,36.80


## 2. Signal test #1 / #2 / #3

I test three signals before using them in a rule.

- **Staleness:** I expect older content to have a stronger refresh need. This is linked to the FlyRank freshness/refresh flag idea.
- **CTR versus position:** I check whether pages with relatively strong positions but weak CTR show a different CTR pattern. This is linked to the CTR-fix idea.
- **Search volume:** I check whether higher-volume content forms a distinct group that could justify prioritization.

Each signal receives one verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE. The verdict is based on the observed bucket results rather than being assumed beforehand.

In [5]:
# 2. Signal test #1 / #2 / #3

# SIGNAL 1: STALENESS
print("=" * 70)
print("SIGNAL TEST #1 — STALENESS")
print("=" * 70)

# Create staleness buckets
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, np.inf],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_test = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean"),
          avg_search_volume=("search_volume", "mean")
      )
      .reset_index()
)

display(staleness_test)

# Compare newer and older content
if len(staleness_test) >= 2:
    first_ctr = staleness_test.iloc[0]["avg_ctr"]
    last_ctr = staleness_test.iloc[-1]["avg_ctr"]

    if last_ctr < first_ctr:
        staleness_verdict = "CONFIRMED"
    elif last_ctr > first_ctr:
        staleness_verdict = "OPPOSITE"
    else:
        staleness_verdict = "MIXED"
else:
    staleness_verdict = "FALSE"

print("Staleness verdict:", staleness_verdict)


# SIGNAL 2: CTR VS POSITION
print("\n" + "=" * 70)
print("SIGNAL TEST #2 — CTR VS POSITION")
print("=" * 70)

# Create position buckets
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-np.inf, 3, 10, 20, np.inf],
    labels=["Top 3", "4-10", "11-20", "21+"]
)

position_test = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean")
      )
      .reset_index()
)

display(position_test)

# Check whether CTR changes across position groups
ctr_values = position_test["avg_ctr"].dropna()

if len(ctr_values) >= 2:
    if ctr_values.iloc[0] > ctr_values.iloc[-1]:
        position_verdict = "CONFIRMED"
    elif ctr_values.iloc[0] < ctr_values.iloc[-1]:
        position_verdict = "OPPOSITE"
    else:
        position_verdict = "MIXED"
else:
    position_verdict = "FALSE"

print("CTR vs position verdict:", position_verdict)


# SIGNAL 3: SEARCH VOLUME
print("\n" + "=" * 70)
print("SIGNAL TEST #3 — SEARCH VOLUME")
print("=" * 70)

# Create search-volume buckets using percentile rank.
# This handles repeated values safely.
volume_rank = df["search_volume"].rank(method="first", pct=True)

df["volume_bucket"] = pd.cut(
    volume_rank,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Low", "Medium-Low", "Medium-High", "High"],
    include_lowest=True
)

volume_test = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("search_volume", "size"),
          avg_impressions=("impressions_90d", "mean"),
          avg_clicks=("clicks_90d", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

display(volume_test)

# Check whether higher search volume corresponds to higher impressions
volume_impressions = volume_test["avg_impressions"].dropna()

if len(volume_impressions) >= 2:
    if volume_impressions.iloc[-1] > volume_impressions.iloc[0]:
        volume_verdict = "CONFIRMED"
    elif volume_impressions.iloc[-1] < volume_impressions.iloc[0]:
        volume_verdict = "OPPOSITE"
    else:
        volume_verdict = "MIXED"
else:
    volume_verdict = "FALSE"

print("Search volume verdict:", volume_verdict)

SIGNAL TEST #1 — STALENESS


,staleness_bucket,n,avg_ctr,avg_position,avg_search_volume
0,0-30 days,20480,0.609021,15.685166,158.441911
1,31-90 days,175,0.117543,16.538286,788.448276
2,91-180 days,9171,0.238367,17.901461,148.585790
3,181+ days,174,3.693276,11.325862,38.000000


Staleness verdict: OPPOSITE

SIGNAL TEST #2 — CTR VS POSITION


,position_bucket,n,avg_ctr,avg_impressions,avg_clicks
0,Top 3,2346,1.472869,3223.757033,15.792413
1,4-10,11842,0.651045,7546.142543,26.340821
2,11-20,7273,0.323443,3137.629589,10.937990
3,21+,8539,0.211333,4247.178241,6.369715


CTR vs position verdict: CONFIRMED

SIGNAL TEST #3 — SEARCH VOLUME


,volume_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,Low,6883,5914.408252,18.780328,0.432920
1,Medium-Low,6883,5667.778440,18.259625,0.387575
2,Medium-High,6883,5062.921110,17.127270,0.267997
3,High,6883,5841.345634,15.325439,0.195332


Search volume verdict: OPPOSITE


## 3. The flag-linked test

The flag-linked signal I test is content staleness. FlyRank's refresh logic is related to content freshness, so I use `days_since_last_update` and compare it with the existing `freshness_tier`.

The purpose is not to assume that the flag is correct. Instead, I check whether the underlying staleness signal actually changes across the existing freshness categories. If the categories show increasing staleness, the signal supports the flag's intended meaning.

In [4]:
# 3. The flag-linked test

print("=" * 70)
print("FLAG-LINKED TEST — FRESHNESS / STALENESS")
print("=" * 70)

# Check the existing freshness flag against its underlying signal
flag_test = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          avg_days_since_update=("days_since_last_update", "mean"),
          median_days_since_update=("days_since_last_update", "median"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

display(flag_test)

# Determine whether freshness tiers generally correspond
# to increasing staleness.
valid_flag_test = flag_test.dropna(
    subset=["avg_days_since_update"]
)

if len(valid_flag_test) >= 2:
    is_increasing = (
        valid_flag_test["avg_days_since_update"]
        .is_monotonic_increasing
    )

    is_decreasing = (
        valid_flag_test["avg_days_since_update"]
        .is_monotonic_decreasing
    )

    if is_increasing or is_decreasing:
        flag_verdict = "CONFIRMED"
    else:
        flag_verdict = "MIXED"
else:
    flag_verdict = "FALSE"

print("Flag-linked verdict:", flag_verdict)

print(
    "\nInterpretation: the verdict above is based on whether "
    "the existing freshness categories show a consistent relationship "
    "with days_since_last_update."
)

FLAG-LINKED TEST — FRESHNESS / STALENESS


,freshness_tier,n,avg_days_since_update,median_days_since_update,avg_ctr,avg_position
0,0-30,20480,18.540918,20.0,0.609021,15.685166
1,181+,174,224.643678,211.0,3.693276,11.325862
2,31-90,175,53.628571,41.0,0.117543,16.538286
3,91-180,9171,104.106204,104.0,0.238367,17.901461


Flag-linked verdict: MIXED

Interpretation: the verdict above is based on whether the existing freshness categories show a consistent relationship with days_since_last_update.


## 4. What this means in practice

The signal checks show which measurable fields are useful for prioritizing content work and which signals should be treated cautiously. Staleness is especially useful because it is directly connected to the freshness/refresh flag, while CTR, position, and search volume provide supporting evidence for prioritization.

In practice, the content team should use these signals as decision-support rather than treating any single signal as proof that a page needs an action.

In [6]:
# 4. What this means in practice

print("Practical conclusion:")
print(
    "The signal audit checks whether staleness, CTR/position, and "
    "search volume provide useful evidence for content prioritization."
)

print(
    "Staleness is especially important because it is linked to the "
    "existing freshness/refresh flag."
)

print(
    "These signals should support a decision rather than automatically "
    "prove that a content action is required."
)

print("\nFinal verdicts:")
print("Staleness:", staleness_verdict)
print("CTR vs position:", position_verdict)
print("Search volume:", volume_verdict)
print("Flag-linked freshness test:", flag_verdict)

Practical conclusion:
The signal audit checks whether staleness, CTR/position, and search volume provide useful evidence for content prioritization.
Staleness is especially important because it is linked to the existing freshness/refresh flag.
These signals should support a decision rather than automatically prove that a content action is required.

Final verdicts:
Staleness: OPPOSITE
CTR vs position: CONFIRMED
Search volume: OPPOSITE
Flag-linked freshness test: MIXED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.